# 04｜单步训练与反向传播

目标：不用完整训练，在 CPU 上观察一次标准的 PyTorch 参数更新。

In [ ]:
from pathlib import Path
import os
PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'model').exists() else Path.cwd().parent
os.chdir(PROJECT_ROOT)
os.environ.setdefault('HF_HOME', str(PROJECT_ROOT / 'learning_notebooks/.cache/huggingface'))
os.environ.setdefault('HF_DATASETS_CACHE', str(PROJECT_ROOT / 'learning_notebooks/.cache/huggingface/datasets'))

import torch
from torch.optim import AdamW
from transformers import AutoTokenizer
from dataset.lm_dataset import PretrainDataset
from model.model_minimind import MiniMindConfig, MiniMindForCausalLM
torch.manual_seed(42)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('./model')
dataset = PretrainDataset(
    'learning_notebooks/samples/pretrain_sample.jsonl',
    tokenizer,
    max_length=32,
)
input_ids, labels = dataset[0]
input_ids = input_ids.unsqueeze(0)
labels = labels.unsqueeze(0)
print('batch 输入：', input_ids.shape)
print('batch 标签：', labels.shape)

In [ ]:
config = MiniMindConfig(hidden_size=64, num_hidden_layers=2)
model = MiniMindForCausalLM(config)
optimizer = AdamW(model.parameters(), lr=1e-3)

tracked_parameter = model.model.layers[0].self_attn.q_proj.weight
before = tracked_parameter.detach().clone()

## 一次完整训练步骤

顺序不能随意交换：清空旧梯度 → 前向传播 → 反向传播 → 更新参数。

In [ ]:
model.train()
optimizer.zero_grad()

output = model(input_ids, labels=labels)
loss = output.loss
print('更新前 loss：', loss.detach().item())

loss.backward()
print('Q 投影梯度范数：', tracked_parameter.grad.norm().item())

torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
optimizer.step()

In [ ]:
after = tracked_parameter.detach().clone()
change = (after - before).abs()
print('参数是否发生变化：', bool(torch.any(change > 0)))
print('最大参数变化量：', change.max().item())

with torch.no_grad():
    new_loss = model(input_ids, labels=labels).loss
print('更新后 loss：', new_loss.item())

## 对照真实训练脚本

现在打开 `trainer/train_pretrain.py` 的 `train_epoch()`，寻找 Notebook 中对应的四步：

1. `model(input_ids, labels=labels)`
2. `loss.backward()`（源码通过 scaler 间接调用）
3. `optimizer.step()`（源码同样通过 scaler 调用）
4. `optimizer.zero_grad()`

## 我的学习记录

- 前向传播与反向传播的区别：
- 梯度存放在哪里？
- `optimizer.step()` 修改了什么？
- 为什么需要 `optimizer.zero_grad()`？
- 一次更新后 loss 的变化是否必然下降？为什么？